In [ ]:
from pathlib import Path
from collections import defaultdict
import os

import cv2
import torch
from IPython.display import FileLink, Video, display
from ultralytics import YOLO

BASE_DIR = Path.cwd()
if (BASE_DIR / "yolov8n.pt").exists() and (BASE_DIR / ".." / "day2" / "video.mp4").exists():
    ROOT = BASE_DIR
else:
    ROOT = BASE_DIR / "week3" / "day6"

VIDEO_PATH = ROOT / ".." / "day2" / "video.mp4"
MODEL_PATH = ROOT / "yolov8n.pt"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH.resolve()}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH.resolve()}")

model = YOLO(str(MODEL_PATH))

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_PATH.resolve()}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

output_dir = ROOT / "runs" / "track" / "day6_trails"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "tracked_with_trails.mp4"

writer = cv2.VideoWriter(
    str(output_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

track_history = defaultdict(list)
frame_count = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model.track(
        frame,
        tracker="bytetrack.yaml",
        persist=True,
        conf=0.25,
        iou=0.5,
        device=DEVICE,
        verbose=False,
    )

    annotated_frame = results[0].plot()

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu()
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            cx, cy, _, _ = box.tolist()
            history = track_history[track_id]
            history.append((int(cx), int(cy)))
            if len(history) > 20:
                history.pop(0)

            for point_a, point_b in zip(history, history[1:]):
                cv2.line(annotated_frame, point_a, point_b, (0, 255, 255), 2)

    writer.write(annotated_frame)
    frame_count += 1

cap.release()
writer.release()

print(f"Processed {frame_count} frames")
print(f"Saved to: {output_path.resolve()}")

output_file = output_path.resolve()
display(FileLink(str(output_file)))
display(Video(str(output_file), embed=True, width=900))
if hasattr(os, "startfile"):
    os.startfile(str(output_file))


Done!
Output saved to: runs/track/day6_trails/tracked_with_trails2.mp4
